# jPCA: pca_dims=3 vs 6

We use 3 PCA dimensions for jPCA.
Original Churchland 2012 method uses 6.
Compare both on v2 checkpoint with noise_at_eval=True.

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    prepare_jpca_input,
    fit_jpca,
    jpca_permutation_test,
    jpca_permutation_test_condition_shuffle,
)

device = "cpu"
print("device:", device)

## 1. Load model

In [ ]:
ckpt_v2 = Path("checkpoints/stage2_v2.pt")
ckpt_v1 = Path("checkpoints/stage2.pt")

if ckpt_v2.exists():
    ckpt_path = ckpt_v2
    sigma_rec = 0.10
    print("Using v2 checkpoint")
else:
    ckpt_path = ckpt_v1
    sigma_rec = 0.05
    print("v2 not found, using v1")


def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=sigma_rec,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model().to(device)
model.load_state_dict(torch.load(str(ckpt_path), weights_only=True)["state_dict"])
model.eval()
model.noise_at_eval = True
print(f"Loaded {ckpt_path.name}, noise_at_eval={model.noise_at_eval}")

## 2. Collect trials

In [ ]:
print("Collecting 5000 trials (with recurrent noise)...")
trials = collect_trials(model, make_env, n_trials=5000, device=device)

outcomes = {}
for tr in trials:
    outcomes[tr["train_outcome"]] = outcomes.get(tr["train_outcome"], 0) + 1
total = len(trials)
print(f"Total: {total}")
for o, n in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")

## 3. Prepare condition-averaged input

In [ ]:
X_cond, labels, rel_time, counts = prepare_jpca_input(
    trials,
    align_key="target_onset",
    window_before=15,
    window_after=30,
    min_trials=10,
    outcome="correct",
)
print(f"X_cond shape: {X_cond.shape}")
print(f"Trials per bin: {counts}")

## 4. jPCA: pca_dims=3 vs pca_dims=6

In [ ]:
results = {}
for dims in [3, 6]:
    res = fit_jpca(X_cond, n_jpcs=1, pca_dims=dims)
    results[dims] = res

    evr = res["pca_pre"].explained_variance_ratio_
    X_flat = X_cond.reshape(-1, X_cond.shape[-1])
    total_var = np.var(X_flat, axis=0).sum()
    Z_flat = res["Z"].reshape(-1, res["Z"].shape[-1])
    jpc_var = np.var(Z_flat, axis=0)
    res["_jpc12_pct"] = jpc_var[:2].sum() / total_var * 100

    print(f"\n=== pca_dims={dims} ===")
    print(f"PCA variance ({dims} dims): {evr.sum()*100:.1f}%")
    print(f"  Per component: {np.round(evr*100, 1)}")
    print(f'var_ratio_r2 (R², primary): {res["var_ratio_r2"]:.3f}  <- always in [0,1]')
    print(f'var_ratio (raw-dX):         {res["var_ratio"]:.3f}  <- can exceed 1')
    print(
        f'var_ratio (Churchland):     {res["var_ratio_churchland"]:.3f}  <- can exceed 1'
    )
    print(f'R2 fit (full M_opt):        {res["r2_fit"]:.3f}')
    print(f'rot_index:              {res["rot_index"]:.3f}')
    print(f"jPC1 var: {jpc_var[0]/total_var*100:.1f}%")
    print(f"jPC2 var: {jpc_var[1]/total_var*100:.1f}%")
    print(f'jPC1+2:   {res["_jpc12_pct"]:.1f}%')

    eigs = res["eigenvalues"]
    print("Top eigenvalues:")
    for i in range(min(3, len(eigs))):
        print(f"  lam{i+1} = {eigs[i].real:+.4f} {eigs[i].imag:+.4f}i")

print("\n" + "=" * 60)

## 5. Comparison table

In [ ]:
print(f'{"Metric":<32} {"pca_dims=3":>12} {"pca_dims=6":>12}')
print("-" * 60)
for key, note in [
    ("var_ratio_r2", "* primary, [0,1]"),
    ("var_ratio", "  raw-dX, can>1"),
    ("var_ratio_churchland", "  Churchland, can>1"),
    ("r2_fit", "  full M_opt R2"),
    ("rot_index", ""),
]:
    v3 = results[3][key]
    v6 = results[6][key]
    print(f"{key:<32} {v3:>12.3f} {v6:>12.3f}  {note}")

print(
    f'{"jPC1+2 (%)":<32} {results[3]["_jpc12_pct"]:>11.1f}% {results[6]["_jpc12_pct"]:>11.1f}%'
)
print()
print("* var_ratio_r2 = R² of skew model vs actual dX: always in [0,1]")
print("  Other var_ratio variants can exceed 1 when M_skew and M_sym")
print("  are anti-correlated in output space (mathematical artifact, not a bug)")

## 6. Trajectory plots: 3D vs 6D

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, dims in zip(axes, [3, 6]):
    Z = results[dims]["Z"]
    for c in range(Z.shape[0]):
        color = plt.cm.plasma(c / (Z.shape[0] - 1))
        ax.plot(Z[c, :, 0], Z[c, :, 1], "-", color=color, lw=1.5, alpha=0.8)
        ax.plot(Z[c, 0, 0], Z[c, 0, 1], "o", color=color, ms=5)
        ax.plot(Z[c, -1, 0], Z[c, -1, 1], "s", color=color, ms=4)

    res = results[dims]
    ax.set_xlabel("jPC1")
    ax.set_ylabel("jPC2")
    ax.set_title(
        f"pca_dims={dims}\n"
        f'var_ratio_r2={res["var_ratio_r2"]:.3f} | '
        f'rot_index={res["rot_index"]:.3f}\n'
        f'jPC1+2={res["_jpc12_pct"]:.1f}%'
    )
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

plt.suptitle("jPCA trajectories: pca_dims=3 vs 6", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Permutation tests (pca_dims=3)

Both methods with pca_dims=3.

In [ ]:
print("=== Permutation tests with pca_dims=3 ===")
print()
print("Time shuffle (100 perms)...")
perm_time_3 = jpca_permutation_test(
    X_cond,
    n_perms=100,
    pca_dims=3,
    n_jpcs=1,
    seed=42,
)
print(
    f'  var_ratio: obs={perm_time_3["real_var_ratio"]:.3f}  '
    f'null={perm_time_3["perm_var_ratios"].mean():.3f}'
    f'+/-{perm_time_3["perm_var_ratios"].std():.3f}  '
    f'p={perm_time_3["p_var_ratio"]:.3f}'
)
print(
    f'  rot_index: obs={perm_time_3["real_rot_index"]:.3f}  '
    f'null={perm_time_3["perm_rot_indices"].mean():.3f}'
    f'+/-{perm_time_3["perm_rot_indices"].std():.3f}  '
    f'p={perm_time_3["p_rot_index"]:.3f}'
)

print()
print("Condition shuffle per neuron (100 perms)...")
perm_paper_3 = jpca_permutation_test_condition_shuffle(
    X_cond,
    n_perms=100,
    pca_dims=3,
    n_jpcs=1,
    seed=42,
)
print(
    f'  var_ratio: obs={perm_paper_3["real_var_ratio"]:.3f}  '
    f'null={perm_paper_3["perm_var_ratios"].mean():.3f}'
    f'+/-{perm_paper_3["perm_var_ratios"].std():.3f}  '
    f'p={perm_paper_3["p_var_ratio"]:.3f}'
)
print(
    f'  rot_index: obs={perm_paper_3["real_rot_index"]:.3f}  '
    f'null={perm_paper_3["perm_rot_indices"].mean():.3f}'
    f'+/-{perm_paper_3["perm_rot_indices"].std():.3f}  '
    f'p={perm_paper_3["p_rot_index"]:.3f}'
)

## 8. Permutation tests (pca_dims=6, for comparison)

In [ ]:
print("=== Permutation tests with pca_dims=6 ===")
print()
print("Time shuffle...")
perm_time_6 = jpca_permutation_test(
    X_cond,
    n_perms=100,
    pca_dims=6,
    n_jpcs=1,
    seed=42,
)
print(
    f'  var_ratio: obs={perm_time_6["real_var_ratio"]:.3f}  '
    f'null={perm_time_6["perm_var_ratios"].mean():.3f}'
    f'+/-{perm_time_6["perm_var_ratios"].std():.3f}  '
    f'p={perm_time_6["p_var_ratio"]:.3f}'
)
print(
    f'  rot_index: obs={perm_time_6["real_rot_index"]:.3f}  '
    f'null={perm_time_6["perm_rot_indices"].mean():.3f}'
    f'+/-{perm_time_6["perm_rot_indices"].std():.3f}  '
    f'p={perm_time_6["p_rot_index"]:.3f}'
)

print()
print("Condition shuffle...")
perm_paper_6 = jpca_permutation_test_condition_shuffle(
    X_cond,
    n_perms=100,
    pca_dims=6,
    n_jpcs=1,
    seed=42,
)
print(
    f'  var_ratio: obs={perm_paper_6["real_var_ratio"]:.3f}  '
    f'null={perm_paper_6["perm_var_ratios"].mean():.3f}'
    f'+/-{perm_paper_6["perm_var_ratios"].std():.3f}  '
    f'p={perm_paper_6["p_var_ratio"]:.3f}'
)
print(
    f'  rot_index: obs={perm_paper_6["real_rot_index"]:.3f}  '
    f'null={perm_paper_6["perm_rot_indices"].mean():.3f}'
    f'+/-{perm_paper_6["perm_rot_indices"].std():.3f}  '
    f'p={perm_paper_6["p_rot_index"]:.3f}'
)

## 9. Summary table: all permutation results

In [ ]:
print()
print("=" * 75)
print(f'{"":<40} {"pca_dims=3":>15} {"pca_dims=6":>15}')
print("-" * 75)

print("Time shuffle:")
print(
    f'  {"var_ratio obs":<38} {perm_time_3["real_var_ratio"]:>15.3f} '
    f'{perm_time_6["real_var_ratio"]:>15.3f}'
)
print(
    f'  {"var_ratio p":<38} {perm_time_3["p_var_ratio"]:>15.3f} '
    f'{perm_time_6["p_var_ratio"]:>15.3f}'
)
print(
    f'  {"rot_index obs":<38} {perm_time_3["real_rot_index"]:>15.3f} '
    f'{perm_time_6["real_rot_index"]:>15.3f}'
)
print(
    f'  {"rot_index p":<38} {perm_time_3["p_rot_index"]:>15.3f} '
    f'{perm_time_6["p_rot_index"]:>15.3f}'
)

print()
print("Condition shuffle:")
print(
    f'  {"var_ratio obs":<38} {perm_paper_3["real_var_ratio"]:>15.3f} '
    f'{perm_paper_6["real_var_ratio"]:>15.3f}'
)
print(
    f'  {"var_ratio p":<38} {perm_paper_3["p_var_ratio"]:>15.3f} '
    f'{perm_paper_6["p_var_ratio"]:>15.3f}'
)
print(
    f'  {"rot_index obs":<38} {perm_paper_3["real_rot_index"]:>15.3f} '
    f'{perm_paper_6["real_rot_index"]:>15.3f}'
)
print(
    f'  {"rot_index p":<38} {perm_paper_3["p_rot_index"]:>15.3f} '
    f'{perm_paper_6["p_rot_index"]:>15.3f}'
)


## 10. Null distribution plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (dims, perm_t, perm_p) in enumerate(
    [
        (3, perm_time_3, perm_paper_3),
        (6, perm_time_6, perm_paper_6),
    ]
):
    ax = axes[0, col]
    ax.hist(
        perm_t["perm_var_ratios"],
        bins=20,
        alpha=0.5,
        color="steelblue",
        label=f'Time shuffle ({perm_t["perm_var_ratios"].mean():.3f})',
    )
    ax.hist(
        perm_p["perm_var_ratios"],
        bins=20,
        alpha=0.5,
        color="coral",
        label=f'Cond shuffle ({perm_p["perm_var_ratios"].mean():.3f})',
    )
    ax.axvline(
        perm_t["real_var_ratio"],
        color="k",
        ls="--",
        lw=2,
        label=f'Observed={perm_t["real_var_ratio"]:.3f}',
    )
    ax.set_title(f"var_ratio  pca_dims={dims}")
    ax.set_xlabel("var_ratio")
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.hist(
        perm_t["perm_rot_indices"],
        bins=20,
        alpha=0.5,
        color="steelblue",
        label=f'Time shuffle ({perm_t["perm_rot_indices"].mean():.3f})',
    )
    ax.hist(
        perm_p["perm_rot_indices"],
        bins=20,
        alpha=0.5,
        color="coral",
        label=f'Cond shuffle ({perm_p["perm_rot_indices"].mean():.3f})',
    )
    ax.axvline(
        perm_t["real_rot_index"],
        color="k",
        ls="--",
        lw=2,
        label=f'Observed={perm_t["real_rot_index"]:.3f}',
    )
    ax.set_title(f"rot_index  pca_dims={dims}")
    ax.set_xlabel("rot_index")
    ax.legend(fontsize=8)

plt.suptitle("Permutation tests: pca_dims=3 (left) vs 6 (right)", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Conclusions

Key questions answered:

1. **pca_dims=3 vs 6**: How do the rotational metrics differ?

2. **var_ratio_churchland vs raw-dX**: Churchland variant always in [0,1].
   Raw-dX can exceed 1.0 with noisy data.

3. **rot_index permutation**: Does pca_dims=3 fix the p=1.0 issue?

4. **jPC1+2 variance**: Does 3D PCA concentrate more variance in the jPC plane?